# Backpropagation Through a Transformer (Q-Former): Step by Step

This notebook traces **exactly how gradients flow backward** through our Q-Former,
using the same tiny matrices from the Q-Former walkthrough.

## The Big Picture

In training, we:
1. **Forward pass**: Audio frames → Q-Former → compressed tokens
2. **Compute loss**: How bad are the tokens? (compare to target)
3. **Backward pass**: Trace the loss backward through every operation
4. **Update**: Nudge each learnable parameter to reduce the loss

The backward pass is just the **chain rule** applied repeatedly:

```
dL/dW = dL/dZ · dZ/dA · dA/dS · dS/dW
        ──────   ─────   ─────   ─────
        from      weighted  softmax  matrix
        loss      sum       deriv    multiply
```

We'll compute every single one of these with actual numbers.

In [1]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=120)

---
## Step 0: Setup Same Example, But With Gradient Tracking

Same as before: 6 encoder frames (dim 4) → 3 queries → 3 compressed tokens.

**difference**: We set `requires_grad=True` on everything we want to train.
The encoder frames F are **frozen** (no grad) , only the queries and projection
weights are trainable.

In [2]:
D = 4   # embedding dimension
T = 6   # encoder frames
m = 3   # queries
scale = math.sqrt(D)

# FROZEN encoder output (Whisper) — no gradients
F_enc = torch.tensor([
    [ 0.1,  0.0,  0.0,  0.1],   # silence
    [ 0.0,  0.1,  0.1,  0.0],   # silence
    [ 0.9,  0.8,  0.2,  0.1],   # "Hel-"
    [ 0.7,  0.9,  0.3,  0.2],   # "-lo"
    [ 0.8,  0.7,  0.5,  0.4],   # "lo" tail
    [ 0.3,  0.2,  0.1,  0.3],   # breath
], requires_grad=False)  # ← FROZEN

# TRAINABLE queries — these will learn
queries = torch.tensor([
    [ 0.5,  0.6,  0.1,  0.0],
    [ 0.1,  0.1,  0.4,  0.5],
    [ 0.3,  0.0,  0.7,  0.2],
], requires_grad=True)  # ← TRAINABLE

# TRAINABLE projection weights
W_q = torch.tensor([
    [ 0.5,  0.3, -0.1,  0.0],
    [ 0.2,  0.6,  0.1, -0.1],
    [-0.1,  0.1,  0.4,  0.2],
    [ 0.0, -0.1,  0.2,  0.5],
], requires_grad=True)  # ← TRAINABLE

W_k = torch.tensor([
    [ 0.4,  0.2,  0.0,  0.1],
    [ 0.1,  0.5,  0.2,  0.0],
    [ 0.0,  0.1,  0.6, -0.1],
    [ 0.1,  0.0, -0.1,  0.4],
], requires_grad=True)  # ← TRAINABLE

W_v = torch.tensor([
    [ 0.6,  0.1,  0.0,  0.1],
    [ 0.0,  0.5,  0.2,  0.0],
    [ 0.1,  0.2,  0.7,  0.0],
    [ 0.0,  0.0,  0.1,  0.6],
], requires_grad=True)  # ← TRAINABLE

print("Trainable parameters:")
print(f"  queries: {queries.shape} ({queries.numel()} values)")
print(f"  W_q:     {W_q.shape} ({W_q.numel()} values)")
print(f"  W_k:     {W_k.shape} ({W_k.numel()} values)")
print(f"  W_v:     {W_v.shape} ({W_v.numel()} values)")
print(f"  Total:   {queries.numel() + W_q.numel() + W_k.numel() + W_v.numel()} trainable values")
print(f"\nFrozen:")
print(f"  F_enc:   {F_enc.shape} ({F_enc.numel()} values) — Whisper output, NOT trained")

Trainable parameters:
  queries: torch.Size([3, 4]) (12 values)
  W_q:     torch.Size([4, 4]) (16 values)
  W_k:     torch.Size([4, 4]) (16 values)
  W_v:     torch.Size([4, 4]) (16 values)
  Total:   60 trainable values

Frozen:
  F_enc:   torch.Size([6, 4]) (24 values) — Whisper output, NOT trained


---
## Step 1: Forward Pass (same as Q-Former notebook)

We do the same cross-attention computation, but now PyTorch is secretly
building a **computation graph** — recording every operation so it can
reverse them during backprop.

```
queries ──→ W_q ──→ Q_proj ──┐
                              ├──→ scores ──→ softmax ──→ weights ──→ Z
F_enc ────→ W_k ──→ K ───────┘                           ↑
       └──→ W_v ──→ V ───────────────────────────────────┘
```

Every arrow is an operation PyTorch records.

In [ ]:
# Step 1a: Project
Q_proj = queries @ W_q     # (3, 4) — projected queries
K = F_enc @ W_k            # (6, 4) — keys
V = F_enc @ W_v            # (6, 4) — values

# Step 1b: Attention scores
scores = Q_proj @ K.T / scale   # (3, 6) — raw scores

# Step 1c: Softmax
attn_weights = F.softmax(scores, dim=-1)   # (3, 6) — probabilities

# Step 1d: Weighted sum → compressed tokens
Z = attn_weights @ V   # (3, 4) — output tokens

print("Forward pass complete!")
print(f"\nQ_proj (3×4):  projected queries")
print(Q_proj)
print(f"\nscores (3×6):  how much each query likes each frame")
print(scores)
print(f"\nattn_weights (3×6):  after softmax (rows sum to 1)")
print(attn_weights)
print(f"\nZ (3×4):  compressed tokens (the output)")
print(Z)

Forward pass complete!

Q_proj (3×4):  projected queries
tensor([[ 0.3600,  0.5200,  0.0500, -0.0400],
        [ 0.0300,  0.0800,  0.2600,  0.3200],
        [ 0.0800,  0.1400,  0.2900,  0.2400]], grad_fn=<MmBackward0>)

scores (3×6):  how much each query likes each frame
tensor([[0.0130, 0.0196, 0.2415, 0.2375, 0.2292, 0.0738],
        [0.0083, 0.0114, 0.0835, 0.0941, 0.1113, 0.0409],
        [0.0080, 0.0150, 0.1124, 0.1227, 0.1372, 0.0457]], grad_fn=<DivBackward0>)

attn_weights (3×6):  after softmax (rows sum to 1)
tensor([[0.1466, 0.1476, 0.1843, 0.1836, 0.1820, 0.1558],
        [0.1584, 0.1589, 0.1708, 0.1726, 0.1756, 0.1637],
        [0.1559, 0.1570, 0.1730, 0.1748, 0.1774, 0.1619]], grad_fn=<SoftmaxBackward0>)

Z (3×4):  compressed tokens (the output)
tensor([[0.3222, 0.3358, 0.2654, 0.1638],
        [0.3086, 0.3210, 0.2556, 0.1601],
        [0.3113, 0.3240, 0.2577, 0.1608]], grad_fn=<MmBackward0>)


---
## Step 2: Define a Target and Compute Loss

To backpropagate, we need a **loss** — a single number that says
"how bad is the output?"

In our real system, the loss would come from contrastive alignment
(Stage 1) or ASR distillation (Stage 2). For this walkthrough,
we use a simple **MSE loss**: we want the compressed tokens to match
a target.

Imagine the target is what a perfect adapter would produce:
- Token 0 should capture the speech content of "Hello"
- Token 1 should capture the energy pattern
- Token 2 should capture timing information

In [4]:
# Target: what we WANT the compressed tokens to be
target = torch.tensor([
    [0.6, 0.5, 0.3, 0.1],   # ideal token 0: speech content
    [0.4, 0.3, 0.5, 0.4],   # ideal token 1: energy pattern
    [0.2, 0.4, 0.6, 0.2],   # ideal token 2: timing
])

# MSE Loss: average squared difference
loss = torch.mean((Z - target) ** 2)

print(f"Output Z:")
print(Z)
print(f"\nTarget:")
print(target)
print(f"\nPer-element error (Z - target):")
print(Z - target)
print(f"\nPer-element squared error:")
print((Z - target) ** 2)
print(f"\nLoss (mean squared error): {loss.item():.6f}")
print(f"\nGoal: make this number smaller by adjusting queries, W_q, W_k, W_v")

Output Z:
tensor([[0.3222, 0.3358, 0.2654, 0.1638],
        [0.3086, 0.3210, 0.2556, 0.1601],
        [0.3113, 0.3240, 0.2577, 0.1608]], grad_fn=<MmBackward0>)

Target:
tensor([[0.6000, 0.5000, 0.3000, 0.1000],
        [0.4000, 0.3000, 0.5000, 0.4000],
        [0.2000, 0.4000, 0.6000, 0.2000]])

Per-element error (Z - target):
tensor([[-0.2778, -0.1642, -0.0346,  0.0638],
        [-0.0914,  0.0210, -0.2444, -0.2399],
        [ 0.1113, -0.0760, -0.3423, -0.0392]], grad_fn=<SubBackward0>)

Per-element squared error:
tensor([[0.0772, 0.0270, 0.0012, 0.0041],
        [0.0084, 0.0004, 0.0597, 0.0576],
        [0.0124, 0.0058, 0.1172, 0.0015]], grad_fn=<PowBackward0>)

Loss (mean squared error): 0.031032

Goal: make this number smaller by adjusting queries, W_q, W_k, W_v


---
## Step 3: Backward Pass (Pytorch)

Calling `loss.backward()` computes **all gradients automatically**.
Each trainable tensor gets a `.grad` attribute: the derivative of the loss
with respect to that tensor.

The gradient tells you: "If I increase this value by a tiny amount,
how much does the loss change?"

- Positive gradient → increasing the value increases the loss (bad) → decrease it
- Negative gradient → increasing the value decreases the loss (good) → increase it

In [5]:
loss.backward()

print("Gradients computed! (dLoss / d_parameter)")
print(f"\n--- dL/d_queries (3×4) ---")
print(f"Shape: {queries.grad.shape}")
print(queries.grad)
print(f"\n--- dL/dW_q (4×4) ---")
print(W_q.grad)
print(f"\n--- dL/dW_k (4×4) ---")
print(W_k.grad)
print(f"\n--- dL/dW_v (4×4) ---")
print(W_v.grad)
print(f"\n--- dL/dF_enc ---")
print(f"F_enc.grad = {F_enc.grad}  ← None because encoder is FROZEN!")

Gradients computed! (dLoss / d_parameter)

--- dL/d_queries (3×4) ---
Shape: torch.Size([3, 4])
tensor([[-0.0013, -0.0017, -0.0006, -0.0002],
        [-0.0009, -0.0012, -0.0005, -0.0002],
        [-0.0007, -0.0009, -0.0004, -0.0002]])

--- dL/dW_q (4×4) ---
tensor([[-0.0011, -0.0016, -0.0009, -0.0003],
        [-0.0010, -0.0014, -0.0008, -0.0003],
        [-0.0011, -0.0017, -0.0010, -0.0004],
        [-0.0007, -0.0010, -0.0006, -0.0002]])

--- dL/dW_k (4×4) ---
tensor([[-0.0012, -0.0019, -0.0011, -0.0009],
        [-0.0012, -0.0019, -0.0011, -0.0009],
        [-0.0005, -0.0008, -0.0006, -0.0005],
        [-0.0002, -0.0004, -0.0003, -0.0003]])

--- dL/dW_v (4×4) ---
tensor([[-0.0216, -0.0182, -0.0501, -0.0170],
        [-0.0209, -0.0176, -0.0484, -0.0164],
        [-0.0092, -0.0077, -0.0215, -0.0073],
        [-0.0081, -0.0069, -0.0194, -0.0067]])

--- dL/dF_enc ---
F_enc.grad = None  ← None because encoder is FROZEN!


---
## Step 4: Now Let's Trace It MANUALLY  Layer by Layer

PyTorch did all the math in one call. Now let's see what it actually
computed at each step. We'll go **backward** through the computation graph:

```
FORWARD:  queries → Q_proj → scores → attn_weights → Z → loss
BACKWARD: queries ← Q_proj ← scores ← attn_weights ← Z ← loss
               ↑        ↑        ↑           ↑        ↑     ↑
              dL/dQ  dL/dQ_p  dL/dS     dL/dA      dL/dZ  dL/dL=1
```

We start from the loss and work backward.

### 4a: dL/dZ — Gradient of loss w.r.t. output tokens

Loss = mean((Z - target)²)

By calculus: dL/dZ = 2(Z - target) / num_elements

This is the **starting gradient** — it tells us how each output value
should change to reduce the loss.

In [6]:
# Manual: derivative of MSE loss w.r.t. Z
num_elements = Z.numel()  # 3 × 4 = 12
dL_dZ = 2 * (Z.detach() - target) / num_elements

print(f"dL/dZ (3×4) — how each output token value should change:")
print(dL_dZ)
print(f"\nInterpretation:")
for i in range(m):
    print(f"  Token {i}: ", end="")
    for j in range(D):
        g = dL_dZ[i, j].item()
        direction = "↓ decrease" if g > 0 else "↑ increase"
        print(f"dim{j}={g:+.4f}({direction})  ", end="")
    print()

dL/dZ (3×4) — how each output token value should change:
tensor([[-0.0463, -0.0274, -0.0058,  0.0106],
        [-0.0152,  0.0035, -0.0407, -0.0400],
        [ 0.0185, -0.0127, -0.0571, -0.0065]])

Interpretation:
  Token 0: dim0=-0.0463(↑ increase)  dim1=-0.0274(↑ increase)  dim2=-0.0058(↑ increase)  dim3=+0.0106(↓ decrease)  
  Token 1: dim0=-0.0152(↑ increase)  dim1=+0.0035(↓ decrease)  dim2=-0.0407(↑ increase)  dim3=-0.0400(↑ increase)  
  Token 2: dim0=+0.0185(↓ decrease)  dim1=-0.0127(↑ increase)  dim2=-0.0571(↑ increase)  dim3=-0.0065(↑ increase)  


### 4b: dL/dA (attention weights) and dL/dV (values)

Z = A @ V, where A = attn_weights (3×6) and V = values (6×4)

By the matrix chain rule:
- **dL/dA = dL/dZ @ V^T** — how should attention weights change?
- **dL/dV = A^T @ dL/dZ** — how should values change? (flows to W_v)

This is the key insight: the gradient flows through **both branches**
of the matrix multiply simultaneously.

In [7]:
A_detached = attn_weights.detach()
V_detached = V.detach()

# dL/dA: (3×4) @ (4×6) = (3×6)
dL_dA = dL_dZ @ V_detached.T

# dL/dV: (6×3) @ (3×4) = (6×4)
dL_dV = A_detached.T @ dL_dZ

print("dL/dA (3×6) — how each attention weight should change:")
print(dL_dA)

labels = ['silence', 'silence', '"Hel-"', '"-lo"', '"lo"tail', 'breath']
print(f"\nInterpretation:")
for i in range(m):
    print(f"  Query {i}:")
    for j in range(T):
        g = dL_dA[i, j].item()
        arrow = "▼" if g > 0 else "▲"
        print(f"    {arrow} Frame {j} ({labels[j]:>8}): {g:+.4f}  "
              f"→ {'attend LESS' if g > 0 else 'attend MORE'}")

print(f"\ndL/dV (6×4) — how each value vector should change:")
print(dL_dV)
print(f"(This gradient flows further back to W_v)")

dL/dA (3×6) — how each attention weight should change:
tensor([[    -0.0024,     -0.0029,     -0.0406,     -0.0371,     -0.0387,     -0.0115],
        [    -0.0041,     -0.0036,     -0.0253,     -0.0291,     -0.0406,     -0.0165],
        [    -0.0000,     -0.0058,     -0.0150,     -0.0236,     -0.0292,     -0.0077]])

Interpretation:
  Query 0:
    ▲ Frame 0 ( silence): -0.0024  → attend MORE
    ▲ Frame 1 ( silence): -0.0029  → attend MORE
    ▲ Frame 2 (  "Hel-"): -0.0406  → attend MORE
    ▲ Frame 3 (   "-lo"): -0.0371  → attend MORE
    ▲ Frame 4 ("lo"tail): -0.0387  → attend MORE
    ▲ Frame 5 (  breath): -0.0115  → attend MORE
  Query 1:
    ▲ Frame 0 ( silence): -0.0041  → attend MORE
    ▲ Frame 1 ( silence): -0.0036  → attend MORE
    ▲ Frame 2 (  "Hel-"): -0.0253  → attend MORE
    ▲ Frame 3 (   "-lo"): -0.0291  → attend MORE
    ▲ Frame 4 ("lo"tail): -0.0406  → attend MORE
    ▲ Frame 5 (  breath): -0.0165  → attend MORE
  Query 2:
    ▲ Frame 0 ( silence): -0.0000  → atten

### 4c: dL/dS (scores)  Backprop Through Softmax

This is the **trickiest step** in transformer backprop.

A = softmax(S), so we need dL/dS = dL/dA · dA/dS.

Softmax is special: changing one score changes ALL the probabilities
(because they must sum to 1). The Jacobian of softmax is:

```
dA_i/dS_j = A_i · (δ_ij - A_j)
```

Where δ_ij = 1 if i==j, else 0.

In practice: `dL/dS_i = A_i · (dL/dA_i - sum_j(A_j · dL/dA_j))`

Let's compute it for each query's row.

In [8]:
# Backprop through softmax, row by row
dL_dS = torch.zeros_like(dL_dA)

for i in range(m):  # for each query
    a_i = A_detached[i]       # (6,) — attention weights for query i
    g_i = dL_dA[i]            # (6,) — incoming gradient for query i

    # The key formula:
    # dL/dS[i] = a_i * (g_i - sum(a_i * g_i))
    dot = torch.sum(a_i * g_i)   # scalar: weighted sum of gradients
    dL_dS[i] = a_i * (g_i - dot)

print("dL/dS (3×6) — gradient of loss w.r.t. raw attention scores:")
print(dL_dS)

print(f"\nWhy softmax backprop is interesting:")
print(f"  Row sums of dL/dS: {dL_dS.sum(dim=-1).tolist()}")
print(f"  They're all ~0! Softmax gradients always sum to zero.")
print(f"  This means: if you attend MORE to one frame, you MUST")
print(f"  attend LESS to others. It's a zero-sum redistribution.")

dL/dS (3×6) — gradient of loss w.r.t. raw attention scores:
tensor([[ 0.0032,  0.0031, -0.0031, -0.0024, -0.0027,  0.0019],
        [ 0.0026,  0.0027, -0.0008, -0.0015, -0.0036,  0.0006],
        [ 0.0022,  0.0013, -0.0002, -0.0017, -0.0027,  0.0010]])

Why softmax backprop is interesting:
  Row sums of dL/dS: [-6.984919309616089e-10, 2.3283064365386963e-10, 1.3969838619232178e-09]
  They're all ~0! Softmax gradients always sum to zero.
  This means: if you attend MORE to one frame, you MUST
  attend LESS to others. It's a zero-sum redistribution.


### 4d: dL/dQ_proj and dL/dK  Backprop Through Score Computation

scores = Q_proj @ K^T / scale

By the matrix chain rule:
- **dL/dQ_proj = dL/dS @ K / scale**
- **dL/dK = dL/dS^T @ Q_proj / scale**

In [9]:
Q_proj_detached = Q_proj.detach()
K_detached = K.detach()

# dL/dQ_proj: (3×6) @ (6×4) / scale = (3×4)
dL_dQ_proj = dL_dS @ K_detached / scale

# dL/dK: (6×3) @ (3×4) / scale = (6×4)
dL_dK = dL_dS.T @ Q_proj_detached / scale

print("dL/dQ_proj (3×4) — how projected queries should change:")
print(dL_dQ_proj)

print(f"\ndL/dK (6×4) — how keys should change:")
print(dL_dK)
print(f"(Keys come from frozen encoder, so this gradient flows to W_k only)")

dL/dQ_proj (3×4) — how projected queries should change:
tensor([[-0.0015, -0.0021, -0.0012, -0.0004],
        [-0.0011, -0.0016, -0.0010, -0.0004],
        [-0.0008, -0.0012, -0.0008, -0.0002]])

dL/dK (6×4) — how keys should change:
tensor([[     0.0007,      0.0011,      0.0007,      0.0006],
        [     0.0006,      0.0010,      0.0006,      0.0005],
        [    -0.0006,     -0.0008,     -0.0002,     -0.0001],
        [    -0.0005,     -0.0008,     -0.0005,     -0.0004],
        [    -0.0006,     -0.0010,     -0.0009,     -0.0008],
        [     0.0004,      0.0006,      0.0003,      0.0002]])
(Keys come from frozen encoder, so this gradient flows to W_k only)


### 4e: dL/dQ (learnable queries) and dL/dW_q  The Final Destination

Q_proj = queries @ W_q

By the matrix chain rule:
- **dL/d_queries = dL/dQ_proj @ W_q^T** — how raw queries should change
- **dL/dW_q = queries^T @ dL/dQ_proj** — how query projection should change

These are the gradients that actually update the learnable parameters!

In [10]:
W_q_detached = W_q.detach()
queries_detached = queries.detach()

# dL/d_queries: (3×4) @ (4×4) = (3×4)
dL_dqueries_from_cross = dL_dQ_proj @ W_q_detached.T

# dL/dW_q: (4×3) @ (3×4) = (4×4)
dL_dW_q_manual = queries_detached.T @ dL_dQ_proj

print("dL/d_queries (3×4) — how learnable queries should change:")
print(dL_dqueries_from_cross)
print(f"\nNote: this is only from the cross-attention path (through Q_proj).")
print(f"The full gradient includes contributions from ALL paths.")

print(f"\ndL/dW_q (4×4) — how query projection weights should change:")
print(dL_dW_q_manual)

dL/d_queries (3×4) — how learnable queries should change:
tensor([[-0.0013, -0.0017, -0.0006, -0.0002],
        [-0.0009, -0.0012, -0.0005, -0.0002],
        [-0.0007, -0.0009, -0.0004, -0.0002]])

Note: this is only from the cross-attention path (through Q_proj).
The full gradient includes contributions from ALL paths.

dL/dW_q (4×4) — how query projection weights should change:
tensor([[-0.0011, -0.0016, -0.0009, -0.0003],
        [-0.0010, -0.0014, -0.0008, -0.0003],
        [-0.0011, -0.0017, -0.0010, -0.0004],
        [-0.0007, -0.0010, -0.0006, -0.0002]])


### 4f: dL/dW_k and dL/dW_v  Frozen Encoder Stops Gradients

K = F_enc @ W_k  and  V = F_enc @ W_v

Even though F_enc is frozen, W_k and W_v are trainable!

- **dL/dW_k = F_enc^T @ dL/dK**
- **dL/dW_v = F_enc^T @ dL/dV**

The gradient reaches W_k and W_v through F_enc, but F_enc itself
doesn't get updated (it has no grad).

In [11]:
# dL/dW_k: (4×6) @ (6×4) = (4×4)
dL_dW_k_manual = F_enc.T @ dL_dK

# dL/dW_v: (4×6) @ (6×4) = (4×4)
dL_dW_v_manual = F_enc.T @ dL_dV

print("dL/dW_k (4×4) — our manual computation:")
print(dL_dW_k_manual)

print(f"\ndL/dW_v (4×4) — our manual computation:")
print(dL_dW_v_manual)

print(f"\nGradient flow diagram:")
print(f"  loss → Z → attn_weights → scores → Q_proj → queries  ✓ (trainable)")
print(f"                                    → Q_proj → W_q      ✓ (trainable)")
print(f"                              scores → K → W_k          ✓ (trainable)")
print(f"                              scores → K → F_enc        ✗ (frozen!)")
print(f"         Z → V → W_v                                    ✓ (trainable)")
print(f"              → F_enc                                   ✗ (frozen!)")

dL/dW_k (4×4) — our manual computation:
tensor([[-0.0012, -0.0019, -0.0011, -0.0009],
        [-0.0012, -0.0019, -0.0011, -0.0009],
        [-0.0005, -0.0008, -0.0006, -0.0005],
        [-0.0002, -0.0004, -0.0003, -0.0003]])

dL/dW_v (4×4) — our manual computation:
tensor([[-0.0216, -0.0182, -0.0501, -0.0170],
        [-0.0209, -0.0176, -0.0484, -0.0164],
        [-0.0092, -0.0077, -0.0215, -0.0073],
        [-0.0081, -0.0069, -0.0194, -0.0067]])

Gradient flow diagram:
  loss → Z → attn_weights → scores → Q_proj → queries  ✓ (trainable)
                                    → Q_proj → W_q      ✓ (trainable)
                              scores → K → W_k          ✓ (trainable)
                              scores → K → F_enc        ✗ (frozen!)
         Z → V → W_v                                    ✓ (trainable)
              → F_enc                                   ✗ (frozen!)


---
## Step 5: Verify  Do Our Manual Gradients Match PyTorch?

PyTorch computed gradients via `loss.backward()`.
We computed them manually step by step. They should match.

In [12]:
print("=" * 60)
print("VERIFICATION: Manual vs PyTorch autograd")
print("=" * 60)

# W_k
match_wk = torch.allclose(dL_dW_k_manual, W_k.grad, atol=1e-5)
print(f"\ndL/dW_k match: {match_wk}")
if not match_wk:
    print(f"  Manual:  {dL_dW_k_manual.flatten()[:4].tolist()}")
    print(f"  PyTorch: {W_k.grad.flatten()[:4].tolist()}")

# W_v
match_wv = torch.allclose(dL_dW_v_manual, W_v.grad, atol=1e-5)
print(f"dL/dW_v match: {match_wv}")

# W_q
match_wq = torch.allclose(dL_dW_q_manual, W_q.grad, atol=1e-5)
print(f"dL/dW_q match: {match_wq}")

# Queries — note: our manual only has the cross-attention path gradient.
# Since this is a simple example with only cross-attention (no self-attention),
# this should match.
match_q = torch.allclose(dL_dqueries_from_cross, queries.grad, atol=1e-5)
print(f"dL/d_queries match: {match_q}")

# F_enc should have no gradient
print(f"F_enc.grad is None: {F_enc.grad is None}  ← frozen encoder, correct!")

all_match = match_wk and match_wv and match_wq and match_q
print(f"\n{'ALL MATCH ✓' if all_match else 'MISMATCH ✗'}")

VERIFICATION: Manual vs PyTorch autograd

dL/dW_k match: True
dL/dW_v match: True
dL/dW_q match: True
dL/d_queries match: True
F_enc.grad is None: True  ← frozen encoder, correct!

ALL MATCH ✓


---
## Step 6: Gradient Descent : Actually Update the Parameters

Now we use the gradients to **update** the parameters.
The simplest update rule: **SGD** (stochastic gradient descent).

```
new_param = old_param - learning_rate × gradient
```

Gradient points "uphill" (toward higher loss), so we go the **opposite
direction** (subtract) to reduce the loss.

In [13]:
lr = 0.1  # learning rate

print("Before update:")
print(f"  queries[0] = {queries.data[0].tolist()}")
print(f"  Loss = {loss.item():.6f}")

# Update each parameter
with torch.no_grad():
    queries_new = queries - lr * queries.grad
    W_q_new = W_q - lr * W_q.grad
    W_k_new = W_k - lr * W_k.grad
    W_v_new = W_v - lr * W_v.grad

print(f"\nAfter update (lr={lr}):")
print(f"  queries[0] = {queries_new[0].tolist()}")
print(f"  Change:      {(-lr * queries.grad)[0].tolist()}")

# Verify: run forward pass with new params and check loss decreased
Q_proj_new = queries_new @ W_q_new
K_new = F_enc @ W_k_new
V_new = F_enc @ W_v_new
scores_new = Q_proj_new @ K_new.T / scale
attn_new = F.softmax(scores_new, dim=-1)
Z_new = attn_new @ V_new
loss_new = torch.mean((Z_new - target) ** 2)

print(f"\nNew loss: {loss_new.item():.6f}")
print(f"Old loss: {loss.item():.6f}")
print(f"Change:   {loss_new.item() - loss.item():+.6f}  "
      f"({'↓ DECREASED' if loss_new < loss else '↑ INCREASED'})")

Before update:
  queries[0] = [0.5, 0.6000000238418579, 0.10000000149011612, 0.0]
  Loss = 0.031032

After update (lr=0.1):
  queries[0] = [0.5001265406608582, 0.6001662611961365, 0.1000615581870079, 2.098081495205406e-05]
  Change:      [0.00012651909491978586, 0.00016621548274997622, 6.155700975796208e-05, 2.098081495205406e-05]

New loss: 0.030225
Old loss: 0.031032
Change:   -0.000807  (↓ DECREASED)


---
## Step 7: Training Loop 

Let's run many iterations and watch the queries **learn** to extract
the right information from the audio frames.

We'll track:
- The loss going down
- The attention patterns changing (queries learning where to look)
- The output tokens approaching the target

In [14]:
# Fresh start
torch.manual_seed(42)

queries_t = torch.tensor([
    [ 0.5,  0.6,  0.1,  0.0],
    [ 0.1,  0.1,  0.4,  0.5],
    [ 0.3,  0.0,  0.7,  0.2],
], requires_grad=True)

W_q_t = torch.tensor([
    [ 0.5,  0.3, -0.1,  0.0],
    [ 0.2,  0.6,  0.1, -0.1],
    [-0.1,  0.1,  0.4,  0.2],
    [ 0.0, -0.1,  0.2,  0.5],
], requires_grad=True)

W_k_t = torch.tensor([
    [ 0.4,  0.2,  0.0,  0.1],
    [ 0.1,  0.5,  0.2,  0.0],
    [ 0.0,  0.1,  0.6, -0.1],
    [ 0.1,  0.0, -0.1,  0.4],
], requires_grad=True)

W_v_t = torch.tensor([
    [ 0.6,  0.1,  0.0,  0.1],
    [ 0.0,  0.5,  0.2,  0.0],
    [ 0.1,  0.2,  0.7,  0.0],
    [ 0.0,  0.0,  0.1,  0.6],
], requires_grad=True)

lr = 0.5
params = [queries_t, W_q_t, W_k_t, W_v_t]
optimizer = torch.optim.SGD(params, lr=lr)

losses = []
attn_history = []

for step in range(200):
    optimizer.zero_grad()

    # Forward
    Q_p = queries_t @ W_q_t
    K_t = F_enc @ W_k_t
    V_t = F_enc @ W_v_t
    s = Q_p @ K_t.T / scale
    a = F.softmax(s, dim=-1)
    z = a @ V_t
    loss_t = torch.mean((z - target) ** 2)

    # Backward
    loss_t.backward()

    # Update
    optimizer.step()

    losses.append(loss_t.item())
    attn_history.append(a.detach().clone())

    if step in [0, 1, 5, 20, 50, 99, 199]:
        print(f"Step {step:3d}: loss = {loss_t.item():.6f}")

print(f"\nFinal loss: {losses[-1]:.6f} (started at {losses[0]:.6f})")
print(f"Reduction:  {(1 - losses[-1]/losses[0])*100:.1f}%")

Step   0: loss = 0.031032
Step   1: loss = 0.027218
Step   5: loss = 0.019458
Step  20: loss = 0.016023
Step  50: loss = 0.015976
Step  99: loss = 0.015967
Step 199: loss = 0.015949

Final loss: 0.015949 (started at 0.031032)
Reduction:  48.6%


---
## Step 8: Visualize  How Attention Patterns Evolved

Let's see how the queries learned to attend to different frames
over training. At the start, attention was roughly uniform.
After training, each query should specialize.

In [15]:
labels = ['silence', 'silence', '"Hel-"', '"-lo"', '"lo"tail', 'breath']
checkpoints = [0, 20, 50, 199]

for step_idx in checkpoints:
    print(f"\n{'='*60}")
    print(f"Step {step_idx} (loss={losses[step_idx]:.6f})")
    print(f"{'='*60}")
    a = attn_history[step_idx]
    for q in range(m):
        print(f"  Query {q}:")
        for f_idx in range(T):
            w = a[q, f_idx].item()
            bar = '█' * int(w * 50)
            print(f"    Frame {f_idx} ({labels[f_idx]:>8}): {w:.3f} {bar}")


Step 0 (loss=0.031032)
  Query 0:
    Frame 0 ( silence): 0.147 ███████
    Frame 1 ( silence): 0.148 ███████
    Frame 2 (  "Hel-"): 0.184 █████████
    Frame 3 (   "-lo"): 0.184 █████████
    Frame 4 ("lo"tail): 0.182 █████████
    Frame 5 (  breath): 0.156 ███████
  Query 1:
    Frame 0 ( silence): 0.158 ███████
    Frame 1 ( silence): 0.159 ███████
    Frame 2 (  "Hel-"): 0.171 ████████
    Frame 3 (   "-lo"): 0.173 ████████
    Frame 4 ("lo"tail): 0.176 ████████
    Frame 5 (  breath): 0.164 ████████
  Query 2:
    Frame 0 ( silence): 0.156 ███████
    Frame 1 ( silence): 0.157 ███████
    Frame 2 (  "Hel-"): 0.173 ████████
    Frame 3 (   "-lo"): 0.175 ████████
    Frame 4 ("lo"tail): 0.177 ████████
    Frame 5 (  breath): 0.162 ████████

Step 20 (loss=0.016023)
  Query 0:
    Frame 0 ( silence): 0.146 ███████
    Frame 1 ( silence): 0.147 ███████
    Frame 2 (  "Hel-"): 0.185 █████████
    Frame 3 (   "-lo"): 0.184 █████████
    Frame 4 ("lo"tail): 0.183 █████████
    Frame 5 (

In [16]:
# Final output vs target
with torch.no_grad():
    Q_final = queries_t @ W_q_t
    K_final = F_enc @ W_k_t
    V_final = F_enc @ W_v_t
    Z_final = F.softmax(Q_final @ K_final.T / scale, dim=-1) @ V_final

print("Final output Z (after training):")
print(Z_final)
print(f"\nTarget:")
print(target)
print(f"\nDifference (Z - target):")
print(Z_final - target)
print(f"\nThe queries learned to extract information that produces")
print(f"tokens close to the target! This is what training does.")

Final output Z (after training):
tensor([[0.4148, 0.4139, 0.4769, 0.2358],
        [0.3959, 0.3944, 0.4569, 0.2287],
        [0.3972, 0.3958, 0.4584, 0.2291]])

Target:
tensor([[0.6000, 0.5000, 0.3000, 0.1000],
        [0.4000, 0.3000, 0.5000, 0.4000],
        [0.2000, 0.4000, 0.6000, 0.2000]])

Difference (Z - target):
tensor([[-0.1852, -0.0861,  0.1769,  0.1358],
        [-0.0041,  0.0944, -0.0431, -0.1713],
        [ 0.1972, -0.0042, -0.1416,  0.0291]])

The queries learned to extract information that produces
tokens close to the target! This is what training does.


---
## Summary: Backprop Through a Transformer

### The Chain of Gradients

```
                     BACKWARD (right to left)
   ←─────────────────────────────────────────────────

   queries ─→ W_q ─→ Q_proj ─┐
   dL/dQ       dL/dW_q        │
                              ├─→ scores ─→ softmax ─→ weights ─→ Z ─→ loss
   F_enc ──→ W_k ─→ K ───────┘    dL/dS                dL/dA    dL/dZ  = 1
   (frozen)  dL/dW_k               (zero-sum)           (two
        └──→ W_v ─→ V ──────────────────────────────→    branches)
             dL/dW_v
```

### Main Operations

| Step | Operation | Gradient rule | Intuition |
|------|-----------|--------------|----------|
| Loss → Z | MSE loss | `2(Z-target)/n` | How far off is each output? |
| Z → weights, V | Matrix multiply | Two branches: `dL/dZ @ V^T` and `A^T @ dL/dZ` | Error splits into "wrong weights" vs "wrong values" |
| weights → scores | Softmax | `A_i · (g_i - Σ A_j g_j)` | Zero-sum: attend more here = attend less there |
| scores → Q_proj, K | Matrix multiply | Two branches again | Error splits into "wrong queries" vs "wrong keys" |
| Q_proj → queries, W_q | Matrix multiply | `dL/dQ_proj @ W_q^T` and `queries^T @ dL/dQ_proj` | Finally reaches learnable params |
| K → W_k (not F_enc) | Matrix multiply | `F_enc^T @ dL/dK` | Gradient reaches W_k but stops at frozen encoder |
